# Analysis of Learned AKOrN Model Dynamics

This notebook analyzes the learned parameters and dynamics of an AKOrN model trained on CIFAR-10 classification.
We will examine:

1. **Omega (Ω)**: The learned natural frequencies/rotational matrices
2. **J**: The learned connectivity/coupling matrices 

Model details:
- Architecture: 3-layer AKOrN with channels [128, 256, 512]
- Oscillator dimension: n=2 (complex oscillators)
- Time steps: T=3 per layer
- No bias in convolution
- Best accuracy: 2% on CIFAR-10

### Jul 5 2025
omega, J analsis moed to a new file (e2025_0705_learned_akorn_parameter_analysis.ipynb)


In [ ]:
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import json
from pathlib import Path
import einops
from einops import rearrange
from sklearn.decomposition import PCA
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Add source directory to path
#sys.path.append('/source')
from models.classification.my_knet import MyAKOrN
from data.augs import augmentation_strong

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Learned Model and Configuration

In [ ]:
# Load the best model checkpoint
checkpoint_path = "results/20250704_570979.opbs/my_akorn_cifar10_final.pth"
config_path = "results/20250704_570979.opbs/parameters.json"

# Load configuration
with open(config_path, 'r') as f:
    config = json.load(f)

print("Model Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)
if 'epoch' in checkpoint_path:
    print(f"\nLoaded checkpoint from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")
elif 'final' in checkpoint_path:
    print(f"\nLoaded final checkpoint with accuracy {checkpoint['final_accuracy']:.2f}%")

# Create model with same configuration
model =MyAKOrN(
    n=config['n'],
    ch=config['ch'], 
    out_classes=config['num_classes'],
    L=config['L'],
    T=config['T'],
    J=config['J'],
    J_bias=config['J_bias'],
    ksizes=config['ksizes'],
    ro_ksize=config['ro_ksize'],
    ro_N=config['ro_N'],
    norm=config['norm'],
    c_norm=config['c_norm'],
    gamma=config['gamma'],
    use_omega=config['use_omega'],
    init_omg=config['init_omg'],
    global_omg=config['global_omg'],
    learn_omg=config['learn_omg'],
    ensemble=config['ensemble']
).to(device)

# Load state dict
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"\nModel loaded successfully!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

## 2. Analysis of Learned Omega Parameters

Omega represents the natural frequencies/rotational dynamics of the oscillators.

In [ ]:
omega_params = []
layer_idx = 0
layer = model.layers[layer_idx]
omega_param = layer[2].omg.omg_param.detach().cpu().numpy()
omega_params.append(omega_param)

In [ ]:
def extract_omega_parameters(model):
    """Extract omega parameters from all layers"""
    omega_params = []
    
    for layer_idx in range(len(model.layers)):
        layer = model.layers[layer_idx]
        if hasattr(layer[2], 'omg') and hasattr(layer[2].omg, 'omg_param'):
            omega_param = layer[2].omg.omg_param.detach().cpu().numpy()
            omega_params.append(omega_param)
            print(f"Layer {layer_idx}: omega shape = {omega_param.shape}")
            print(f"  Omega values: {omega_param}")
            print(f"  Omega magnitude: {np.linalg.norm(omega_param):.4f}")
    
    return omega_params

omega_params = extract_omega_parameters(model)


In [ ]:
fig, axes = plt.subplots(1, len(model.layers), figsize=(15, 4))
for layer_idx in range(len(model.layers)):
    omega = omega_params[layer_idx][:, 0]  # Take the first column (identical to second)
    axes[layer_idx].hist(omega, bins=20, color='C0', alpha=0.7)
    axes[layer_idx].set_title(f'Layer {i} Omega Histogram')
    axes[layer_idx].set_xlabel('Omega Value')
    axes[layer_idx].set_ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Analysis of Learned Connectivity Matrices (J)

The connectivity matrices determine how oscillators couple with each other.

In [ ]:
def extract_connectivity_weights(model):
    """Extract connectivity weight matrices from all layers"""
    connectivity_weights = []
    
    for layer_idx in range(len(model.layers)):
        layer = model.layers[layer_idx]
        if hasattr(layer[2], 'connectivity'):
            weight = layer[2].connectivity.weight.detach().cpu().numpy()
            bias = layer[2].connectivity.bias.detach().cpu().numpy() if layer[2].connectivity.bias is not None else None
            
            connectivity_weights.append({
                'weight': weight,
                'bias': bias,
                'shape': weight.shape
            })
            
            print(f"Layer {layer_idx}: Connectivity weight shape = {weight.shape}")
            print(f"  Weight statistics: mean={weight.mean():.4f}, std={weight.std():.4f}")
            print(f"  Weight range: [{weight.min():.4f}, {weight.max():.4f}]")
            if bias is not None:
                print(f"  Bias statistics: mean={bias.mean():.4f}, std={bias.std():.4f}")
    
    return connectivity_weights

connectivity_weights = extract_connectivity_weights(model)


## $J_{ij}$'s as 2x2 matrices
Here we are to have a close look at the connectivity matrices $J$.

Take the first layer for an example. Here, the whole weight tensor looks like $(n_{\text{output ch}}, n_{\text{input ch}}, H, W)$. For instance, it would be of the shape $(128,128,9,9)$, where the firt corrdinate is the ouptout oscillator that is influenced, the second the input oscillator that gives influence to the first, and the latter two determine the location in the 9x9 kernel. Considering the dimension of the oscillators is two, to see the connectivity $J_{ij}$, or the influence from $j$ to $i$, we are to check the index $[2i:2i+1, 2j:2j+1]$.

We, therefore, have 64x64x9x9 = 3.3e5 connectivity matrices per layer. This is far too many, so we must see summarized statistics of the connectivity matrices. To do so, we first examine the **Strength of the connectivity**, i.e., $\|J_{ij}\|_F$.


In [ ]:
# Let J = connectivity_weights[0]['weight']. 
# Can you draw a 9x9 subplot, where ij'th subplot is a 64x64 matrix whose kl'th element is the Frobenius norm of J[2k:2k+2, 2l:2l+2, i, j]? 
# Use the same color limit for all subplot. Add titles and suptitle if necessary.
J = connectivity_weights[0]['weight']
fig, axes = plt.subplots(9, 9, figsize=(18, 18))
C_out, C_in, H, W = J.shape
matrices = np.zeros((9, 9, C_out // 2, C_in // 2))

# Compute all matrices and find global vmin/vmax
for i in range(9):
    for j in range(9):
        for k in range(C_out // 2):
            for l in range(C_in // 2):
                block = J[2*k:2*k+2, 2*l:2*l+2, i, j]
                matrices[i, j, k, l] = np.linalg.norm(block, ord='fro')
vmin = matrices.min()
vmax = matrices.max()

for i in range(9):
    for j in range(9):
        ax = axes[i, j]
        im = ax.imshow(matrices[i, j], cmap='viridis', vmin=vmin, vmax=vmax)
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f'({i},{j})', fontsize=8)
plt.suptitle('Frobenius Norms of $J_{ij}$ Blocks (Each subplot: 64x64)', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.97])
cbar = fig.colorbar(im, ax=axes, orientation='vertical', fraction=0.01, pad=0.01)
plt.show()

In [ ]:
# Compute and visualize summary statistics (mean, std, min, max) for each 64x64 matrix in the 9x9 grid

for l in range(config['L']):

    J = connectivity_weights[l]['weight']
    C_out, C_in, H, W = J.shape
    matrix_shape = (C_out // 2, C_in // 2)

    # Precompute statistics for each (i, j) kernel position
    means = np.zeros((H, W))
    stds = np.zeros((H, W))
    mins = np.zeros((H, W))
    maxs = np.zeros((H, W))

    for i in range(H):
        for j in range(W):
            mat = np.zeros(matrix_shape)
            for k in range(matrix_shape[0]):
                for l in range(matrix_shape[1]):
                    block = J[2*k:2*k+2, 2*l:2*l+2, i, j]
                    mat[k, l] = np.linalg.norm(block, ord='fro')
            means[i, j] = mat.mean()
            stds[i, j] = mat.std()
            mins[i, j] = mat.min()
            maxs[i, j] = mat.max()

    # Plot all statistics in a 2x2 grid of 9x9 subplots
    fig, axes = plt.subplots(2, 2, figsize=(10, 8))
    stat_titles = ['Mean', 'Std', 'Min', 'Max']
    stat_arrays = [means, stds, mins, maxs]
    cmaps = ['viridis', 'magma', 'Blues', 'Reds']

    for idx, (ax, stat, title, cmap) in enumerate(zip(axes.flat, stat_arrays, stat_titles, cmaps)):
        im = ax.imshow(stat, cmap=cmap)
        ax.set_title(f'{title} of {matrix_shape[0]}x{matrix_shape[1]} Block ({H}x{W} kernels)', fontsize=14)
        ax.set_xlabel('Kernel W')
        ax.set_ylabel('Kernel H')
        ax.grid(False)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.suptitle(f'Summary Statistics for Frobenius norms of {l+1} Oscillators ({H}x{W} Kernel Grid)', fontsize=18)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

In [ ]:
# # This time, can you draw a 64x64 subplot, where kl'th subplot is a 9x9 matrix whose ij'th element is the Frobenius norm of J[2k:2k+2, 2l:2l+2, i, j]? 
# # Use the same color limit for all subplot. Add titles and suptitle if necessary.
# # Assume J = connectivity_weights[0]['weight'] (shape: [128, 128, 9, 9])
# J = connectivity_weights[0]['weight']
# C_out, C_in, H, W = J.shape
# n = 2  # oscillator dimension

# # Number of oscillators per channel
# num_out = C_out // n
# num_in = C_in // n

# # Compute the 64x64 grid of 9x9 matrices (each entry is a 9x9 Frobenius norm map)
# frobenius_maps = np.zeros((num_out, num_in, H, W))
# for k in range(num_out):
#     for l in range(num_in):
#         for i in range(H):
#             for j in range(W):
#                 block = J[n*k:n*k+n, n*l:n*l+n, i, j]
#                 frobenius_maps[k, l, i, j] = np.linalg.norm(block, ord='fro')

# # Find global vmin/vmax for consistent color scale
# vmin = frobenius_maps.min()
# vmax = frobenius_maps.max()

# fig, axes = plt.subplots(num_out, num_in, figsize=(num_in, num_out), dpi=120)
# if num_out == 1 and num_in == 1:
#     axes = np.array([[axes]])
# elif num_out == 1 or num_in == 1:
#     axes = axes.reshape(num_out, num_in)

# for k in range(num_out):
#     for l in range(num_in):
#         ax = axes[k, l]
#         im = ax.imshow(frobenius_maps[k, l], cmap='viridis', vmin=vmin, vmax=vmax)
#         ax.set_xticks([])
#         ax.set_yticks([])
#         if k == 0:
#             ax.set_title(f'in {l}', fontsize=6)
#         if l == 0:
#             ax.set_ylabel(f'out {k}', fontsize=6)

# plt.suptitle('Each subplot: 9x9 Frobenius norm map for J[2k:2k+2, 2l:2l+2, :, :]', fontsize=12)
# plt.tight_layout(rect=[0, 0, 1, 0.97])
# fig.colorbar(im, ax=axes.ravel().tolist(), orientation='vertical', fraction=0.01, pad=0.01)
# plt.show()

In [ ]:
# Compute and visualize summary statistics (mean, std, min, max) for each 9x9 matrix in the 64x64 grid

In [ ]:
# Compute and visualize summary statistics (mean, std, min, max) for each 9x9 matrix in the 64x64 grid
for lay in range(config['L']):

    J = connectivity_weights[lay]['weight']
    C_out, C_in, H, W = J.shape
    n = 2  # oscillator dimension (already defined above)
    num_out = C_out // n
    num_in = C_in // n
    matrix_shape = (num_out, num_in)
    # frobenius_maps: shape [num_out, num_in, H, W] (from cell 14)
    # Each [k, l] is a 9x9 matrix

    mean_grid = np.zeros((num_out, num_in))
    std_grid = np.zeros((num_out, num_in))
    min_grid = np.zeros((num_out, num_in))
    max_grid = np.zeros((num_out, num_in))

    # Compute the 64x64 grid of 9x9 matrices (each entry is a 9x9 Frobenius norm map)
    frobenius_maps = np.zeros((num_out, num_in, H, W))
    for k in range(num_out):
        for l in range(num_in):
            for i in range(H):
                for j in range(W):
                    block = J[n*k:n*k+n, n*l:n*l+n, i, j]
                    frobenius_maps[k, l, i, j] = np.linalg.norm(block, ord='fro')


    for k in range(num_out):
        for l in range(num_in):
            mat = frobenius_maps[k, l]
            mean_grid[k, l] = mat.mean()
            std_grid[k, l] = mat.std()
            min_grid[k, l] = mat.min()
            max_grid[k, l] = mat.max()

    stat_grids = [mean_grid, std_grid, min_grid, max_grid]
    stat_titles = ['Mean', 'Std', 'Min', 'Max']
    cmaps = ['viridis', 'magma', 'Blues', 'Reds']

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    for idx, (ax, grid, title, cmap) in enumerate(zip(axes.flat, stat_grids, stat_titles, cmaps)):
        im = ax.imshow(grid, cmap=cmap)
        ax.set_title(f'{title} of {H}x{W} Frobenius Norms ({matrix_shape[0]}x{matrix_shape[1]} grid)')
        ax.set_xlabel('Input Oscillator')
        ax.set_ylabel('Output Oscillator')
        ax.grid(False)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.suptitle(f'Summary Statistics for Each {H}x{W} Matrix in {matrix_shape[0]}x{matrix_shape[1]} Grid', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

## How many clusters of J_ij?

In [ ]:
# Let J = connectivity_weights[0]['weight'].
# J = connectivity_weights[0]['weight']という変数を[128, 128, 9, 9]型torch.tensorとする。
# 各i, j, k, lに対してJ[2k:2k+2, 2l:2l+2, i, j] を上でいった2x2の行列とする。
# 実装例で示してくれたみたいに4次元行列にflattenしたときのユークリッド距離でクラスターを見たいんだけど、今言った状況で実装してみてくれない？

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# ------------------------------------------------------------
# 0. 前提: J は [128,128,9,9] の torch.Tensor
# ------------------------------------------------------------
J = connectivity_weights[0]['weight']          # dtype=float32/64 など想定
assert J.shape == (128, 128, 9, 9)

# ------------------------------------------------------------
# 1. 2×2 ブロックを全部取り出す
#    reshape → transpose で (k,l,i,j,2,2) = (64,64,9,9,2,2)
# ------------------------------------------------------------
blocks = (
    J.reshape(64, 2, 64, 2, 9, 9)      # (64,2,64,2,9,9)
     .transpose(0, 2, 4, 5, 1, 3)      # (64,64,9,9,2,2)
     .reshape(-1, 2, 2)                # (331_776, 2, 2)
)

# ------------------------------------------------------------
# 2. vec(A) = (a11,a12,a21,a22) に flatten して NumPy へ
# ------------------------------------------------------------
X = blocks.reshape(blocks.shape[0], -1)    # (331_776, 4)


In [ ]:
import pandas as pd

# Draw a violin plot of the four sets of elements X[:, ii] for ii=0 to 3

df_X = pd.DataFrame({
    'a11': X[:, 0],
    'a12': X[:, 1],
    'a21': X[:, 2],
    'a22': X[:, 3]
})

plt.figure(figsize=(8, 5))
sns.violinplot(data=df_X, inner='quartile')
plt.title('Distribution of 2x2 Block Elements')
plt.ylabel('Value')
plt.xlabel('Matrix Entry')
plt.tight_layout()
plt.show()

In [ ]:
# Compute the Frobenius norm of each row of block (i.e., block[ii, :, :]) for all ii
# block is assumed to be a 3D numpy array of shape (N, 2, 2)

frob_norms = np.linalg.norm(blocks, axis=(1, 2))  # Frobenius norm for each ii

plt.figure(figsize=(6, 4))
plt.hist(frob_norms, bins=80, color='C0', alpha=0.7)
plt.title('Distribution of Frobenius Norms of block[ii, :, :]')
plt.xlabel('Frobenius Norm')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Draw the distribution of the Frobenius norm of block[ii, :, :]
# Assume block is a 3D numpy array of shape (N, 2, 2), where N >= ii+1

def frobenius_norm_distribution(block, ii):
    """
    Plot the distribution of the Frobenius norm of block[ii, :, :].
    """
    block_ii = block[ii, :, :]
    frob_norm = np.linalg.norm(block_ii, ord='fro')
    plt.figure(figsize=(5, 3))
    plt.hist([frob_norm], bins=20, color='C0', alpha=0.7)
    plt.title(f'Frobenius Norm Distribution of block[{ii}, :, :]')
    plt.xlabel('Frobenius Norm')
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

# Example usage:
frobenius_norm_distribution(block, ii)

In [ ]:

# ------------------------------------------------------------
# 3. z-スコア標準化（列ごとに平均0, 分散1）
# ------------------------------------------------------------
#X = StandardScaler().fit_transform(X)

# ------------------------------------------------------------
# 4. k を 2‥10 で総当たり → シルエット最大を採択
# ------------------------------------------------------------
from tqdm import tqdm

best_k, best_score = None, -1
for k in tqdm(range(2, 11)):
    km = KMeans(n_clusters=k, n_init='auto', random_state=0).fit(X)
    score = silhouette_score(X, km.labels_, sample_size=10_000)
    if score > best_score:
        best_k, best_score, best_model = k, score, km
print(f"採択: k={best_k}, silhouette={best_score:.3f}")

labels  = best_model.labels_                      # (331_776,)
centers = best_model.cluster_centers_.reshape(best_k, 2, 2)

# ------------------------------------------------------------
# 5. ざっと結果を眺める
# ------------------------------------------------------------
for k in range(best_k):
    cnt = (labels == k).sum()
    print(f"Cluster {k}: {cnt:6d} mats  |  center =\n{centers[k]}")


In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
pca = PCA(n_components=3, random_state=0)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(6,5))
plt.scatter(X_pca[:,0], X_pca[:,1], s=5, c=labels, cmap='tab10')
plt.title('PCA (2-D) of 2×2 blocks')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.tight_layout(); plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # 明示的に import（必須）
from mpl_toolkits.mplot3d import proj3d  # 投影処理用（通常不要）

fig = plt.figure(figsize=(6,5))
ax = fig.add_subplot(111, projection='3d')

# 3D 座標 (N, 3)、色ラベル (N,)
ax.scatter(X_pca[:,0], X_pca[:,1], X_pca[:,2], c=labels, s=5, cmap='tab10')

ax.set_title("3D Scatter")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
plt.tight_layout()
plt.show()

In [ ]:
N          = X.shape[0]
rng        = np.random.default_rng(0)
idx_vis    = rng.choice(N, size=20_000, replace=False)
X_vis      = X[idx_vis]
labels_vis = labels[idx_vis]

tsne = TSNE(
    n_components=2,
    perplexity=30,
    init='pca',
    learning_rate='auto',
    random_state=0,
)
X_tsne = tsne.fit_transform(X_vis)

plt.figure(figsize=(6,5))
plt.scatter(X_tsne[:,0], X_tsne[:,1], s=5, c=labels_vis, cmap='tab10')
plt.title('t-SNE of 2×2 blocks')
plt.xlabel('dim-1'); plt.ylabel('dim-2')
plt.tight_layout(); plt.show()

In [ ]:
import umap

# ------------------------------------------------------------
# 1. 描画用にランダム・サブサンプル（推奨 1–2 万点）
# ------------------------------------------------------------
N          = X.shape[0]
rng        = np.random.default_rng(0)
idx_vis    = rng.choice(N, size=20_000, replace=False)
X_vis      = X[idx_vis]
labels_vis = labels[idx_vis]

# ------------------------------------------------------------
# 2. UMAP (2 次元) で埋め込み
#    ・n_neighbors: 近傍のスケール (5–50 でチューニング)
#    ・min_dist   : 点をどれだけ詰めるか (0–0.5)
# ------------------------------------------------------------
mapper = umap.UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.1,
    metric="euclidean",
    random_state=0,
)
X_umap = mapper.fit_transform(X_vis)   # (20_000, 2)

# ------------------------------------------------------------
# 3. 可視化
# ------------------------------------------------------------
plt.figure(figsize=(6,5))
plt.scatter(
    X_umap[:, 0], X_umap[:, 1],
    s=5, c=labels_vis, cmap="tab10", alpha=0.8
)
plt.title("UMAP of 2×2 blocks")
plt.xlabel("UMAP-1"); plt.ylabel("UMAP-2")
plt.tight_layout()
plt.show()

## Decompose $J_{ij}$ into a sum of symmetric + skew symmetric part

$$
J_{ij} = J_{ij}^{\textrm{Sym}} + J_{ij}^{\textrm{Skew}} = \begin{pmatrix} p^1_{ij} & p^2_{ij} \\ p^2_{ij} & p^3_{ij}  \end{pmatrix} + \begin{pmatrix} 0 & -q_{ij} \\ q_{ij} & 0  \end{pmatrix} 

In [ ]:
def decompose_sym_skew(J):
    """
    Decompose a 2x2 matrix or a batch of n 2x2 matrices into rotation and symmetric parts.

    Args:
        J: shape (2,2) or (n,2,2) numpy array

    Returns:
        c_R: (n,) array or scalar
        c_S: (n,) array or scalar
        alpha: (n,) array or scalar
        beta: (n,) array or scalar
    """
    J = np.asarray(J)
    J_sym  = (J + J.T) / 2
    J_skew = (J - J.T) / 2
    if J.ndim == 2 and J.shape == (2, 2):
        p1, p2, p3 = J_sym[0,0], J_sym[0,1], J_sym[1,1]
        q = J_skew[1,0]
    elif J.ndim == 3 and J.shape[1:] == (2, 2):
        p1 = J[:, 0, 0]
        p2 = J[:, 0, 1]
        p3 = J[:, 1, 1]
        q  = J[:, 1, 0]
    else:
        raise ValueError("Input must be shape (2,2) or (n,2,2)") 
    return p1, p2, p3, q
    

## Decompose $J_{ij}$ into a sum of rotation and symmetric part

Acording to Costa \& Aguiar 2024 and Buzanello, Barioni, \& Aguiar 2022, if the connectivity matrix $J_{ij}$ can be decomposed as

\begin{equation}
J_{ij}= c^{\text{R}}_{ij}\left(\begin{array}{cc}
\cos \alpha_{ij} & \sin \alpha_{ij} \\
-\sin \alpha_{ij} & \cos \alpha_{ij}
\end{array}\right)
+ c^{\text{S}}_{ij}\left(\begin{array}{cc}
-\cos \beta_{ij} & \sin \beta_{ij} \\
\sin \beta_{ij} & \cos \beta_{ij}
\end{array}\right),
\end{equation}

then the equation of $\theta_i$ becomes
\begin{equation}
\dot{\theta}_i = \omega_i + \sum_{j=i}^n c^{\text{R}}_{ij} \sin (\theta_j - \theta_i - \alpha)
+c^{\text{S}}_{ij} \sin (\theta_j + \theta_i + \beta) 
\end{equation}

In [ ]:
def decompose_RS(J):
    """
    Decompose a 2x2 matrix or a batch of n 2x2 matrices into rotation and symmetric parts.

    Args:
        J: shape (2,2) or (n,2,2) numpy array

    Returns:
        c_R: (n,) array or scalar
        c_S: (n,) array or scalar
        alpha: (n,) array or scalar
        beta: (n,) array or scalar
    """
    J = np.asarray(J)
    if J.ndim == 2 and J.shape == (2, 2):
        a, b = J[0, 0], J[0, 1]
        c, d = J[1, 0], J[1, 1]
    elif J.ndim == 3 and J.shape[1:] == (2, 2):
        a = J[:, 0, 0]
        b = J[:, 0, 1]
        c = J[:, 1, 0]
        d = J[:, 1, 1]
    else:
        raise ValueError("Input must be shape (2,2) or (n,2,2)") 
    c_R = 0.5 * np.sqrt((a + d)**2 + (b - c)**2)
    c_S = 0.5 * np.sqrt((d - a)**2 + (b + c)**2)
    alpha = np.arctan2(b - c, a + d)
    beta  = np.arctan2(b + c, d - a)
    return c_R, c_S, alpha, beta
    

In [ ]:
def rotation_matrix(alpha):
    """
    Returns the 2x2 rotation matrix for angle alpha.
    """
    return np.array([
        [np.cos(alpha), -np.sin(alpha)],
        [np.sin(alpha),  np.cos(alpha)]
    ])

def reflection_matrix(alpha):
    """
    Returns the 2x2 reflection matrix about the line making angle alpha/2 with the x-axis.
    """
    return np.array([
        [np.cos(alpha),  np.sin(alpha)],
        [np.sin(alpha), -np.cos(alpha)]
    ])

In [ ]:
RR = np.random.rand(1,2,2)
print(RR)


In [ ]:
c_R, c_S, alpha, beta = decompose_RS(RR)
print(c_R.shape)
print(alpha.shape)

In [ ]:

RR_tobe = c_R * rotation_matrix(-alpha) + c_S * reflection_matrix(np.pi-beta)
print(RR_tobe)

In [ ]:
J = connectivity_weights[0]['weight']          # dtype=float32/64 など想定
# (n * n_osc, n * n_osc, ksize, ksize)
C_out, C_in, ksize, _ = J.shape

# ------------------------------------------------------------
# 1. 2×2 ブロックを全部取り出す
#    reshape → transpose で (k,l,i,j,2,2) = (64,64,9,9,2,2)
# ------------------------------------------------------------
J_blocks = (
    J.reshape(C_out//n, n, C_in//n, n, ksize, ksize)      # (64,2,64,2,9,9)
     .transpose(0, 2, 4, 5, 1, 3)      # (64,64,9,9,2,2)
     .reshape(-1, 2, 2)                # (331_776, 2, 2)
)

c_R, c_S, alpha, beta = decompose_RS(J_blocks)

In [ ]:
# Reshape c_R, c_S, alpha, beta back to (num_out, num_in, ksize, ksize)
C_out, C_in, ksize, _ = J.shape  # J is already defined above
shape = (C_out//n, C_in//n, ksize, ksize)

c_R_reshaped = c_R.reshape(shape)
c_S_reshaped = c_S.reshape(shape)
alpha_reshaped = alpha.reshape(shape)
beta_reshaped = beta.reshape(shape)

In [ ]:
import pandas as pd

# Flatten c_R_reshaped, c_S_reshaped, alpha_reshaped, beta_reshaped for violin plots
c_R_flat = c_R_reshaped.flatten()
c_S_flat = c_S_reshaped.flatten()
alpha_flat = alpha_reshaped.flatten()
beta_flat = beta_reshaped.flatten()

# Prepare data for seaborn violinplot

df_violin = pd.DataFrame({
    'c_R': c_R,
    'c_S': c_S,
    'alpha': alpha,
    'beta': beta
})

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# c_R and c_S
sns.violinplot(data=df_violin[['c_R', 'c_S']], ax=axes[0], inner='quartile')
axes[0].set_title('Distribution of $c_R$ and $c_S$')
axes[0].set_ylabel('Value')

# alpha and beta
sns.violinplot(data=df_violin[['alpha', 'beta']], ax=axes[1], inner='quartile')
axes[1].set_title('Distribution of $\\alpha$ and $\\beta$')
axes[1].set_ylabel('Angle (radians)')
axes[1].set_xticklabels([r'$\alpha$', r'$\beta$'])

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(c_R, c_S, alpha=0.3, s=8)
plt.xlabel('$c_R$')
plt.ylabel('$c_S$')
plt.title('Scatter plot of $c_R$ vs $c_S$')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(alpha, beta, alpha=0.3, s=8)
plt.xlabel('$alpha$')
plt.ylabel('$beta$')
plt.title('Scatter plot of $alpha$ vs $beta$')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
from matplotlib import cm
# --- 1. 角度を [0, 2π) に並べ替える（周期をそろえるだけ） --------------------
theta = (alpha + np.pi) % (2*np.pi)   # toroidal angle (大円周り)
phi   = (beta  + np.pi) % (2*np.pi)   # poloidal angle (小円周り)

# --- 2. 2D 密度ヒストグラム ---------------------------------------------------
n_bins = 100
H, theta_edges, phi_edges = np.histogram2d(theta, phi,
                                           bins=n_bins,
                                           range=[[0, 2*np.pi], [0, 2*np.pi]],
                                           density=True)

# ①：とりあえず平面ヒートマップ（周期境界付き）
plt.figure(figsize=(4.5, 4))
plt.imshow(H.T, origin='lower', cmap='viridis',
           extent=[-np.pi, np.pi, -np.pi, np.pi], aspect='auto')
plt.xlabel(r'$\alpha$ (toroidal)')
plt.ylabel(r'$\beta$ (poloidal)')
plt.colorbar(label='density')
plt.grid(False)
plt.tight_layout()
plt.show()

# --- 3. トーラスへ貼る --------------------------------------------------------
# パラメータ（好みに応じて調整）
R, r = 1.1, 1.0   # R: 大半径, r: 小半径
# ヒストグラム bin 中心を座標に変換
theta_c = 0.5 * (theta_edges[:-1] + theta_edges[1:])
phi_c   = 0.5 * (phi_edges  [:-1] + phi_edges  [1:])
Theta, Phi = np.meshgrid(theta_c, phi_c, indexing='ij')   # (n_bins, n_bins)

X = (R + r*np.cos(Phi)) * np.cos(Theta)
Y = (R + r*np.cos(Phi)) * np.sin(Theta)
Z =  r*np.sin(Phi)

# カラースケール：密度を 0-1 の範囲に正規化
H_norm = (H - H.min()) / (H.max() - H.min() + 1e-9)
colors = cm.viridis(H_norm)

fig = plt.figure(figsize=(6, 5.5))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, rstride=1, cstride=1,
                facecolors=colors, linewidth=0, antialiased=False,
                shade=False)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# ① まだなら一度だけ
%pip install -q ipympl
# ② セッションごとに
%load_ext ipympl
%matplotlib widget

fig = plt.figure(figsize=(6, 5))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, Z, rstride=1, cstride=1,
                facecolors=cm.viridis(H_norm), linewidth=0, antialiased=False)
ax.set_axis_off()
plt.show()        

In [ ]:
%matplotlib inline
from mpl_toolkits.mplot3d import Axes3D

# Torus parameters
R = 3.0  # Major radius
r = 0.5  # Minor radius

# Create torus mesh for surface
u = np.linspace(0, 2 * np.pi, 100)
v = np.linspace(0, 2 * np.pi, 40)
U, V = np.meshgrid(u, v)
X_surf = (R + r * np.cos(V)) * np.cos(U)
Y_surf = (R + r * np.cos(V)) * np.sin(U)
Z_surf = r * np.sin(V)

# Normalize alpha and beta to [0, 2pi)
alpha_mod = np.mod(alpha, 2 * np.pi)
beta_mod = np.mod(beta, 2 * np.pi)

# Data points
X = (R + r * np.cos(beta_mod)) * np.cos(alpha_mod)
Y = (R + r * np.cos(beta_mod)) * np.sin(alpha_mod)
Z = r * np.sin(beta_mod)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')

# Plot torus surface
ax.plot_surface(X_surf, Y_surf, Z_surf, color='wheat', alpha=0.7, edgecolor='none', zorder=1, shade=True)

# Overlay points
ax.scatter(X, Y, Z, c=alpha_mod, cmap='twilight', s=6, alpha=0.5, zorder=2)

# Make it look like a donut
ax.set_box_aspect([1,1,0.5])
ax.set_axis_off()
ax.view_init(elev=30, azim=45)
ax.set_title('Distribution of $(\\alpha, \\beta)$ on a Torus', pad=20, fontsize=16)
plt.tight_layout()
plt.show()


In [ ]:
# c_R, c_S, alpha, beta are currently flat arrays of length (C_out//n) * (C_in//n) * ksize * ksize
# Reshape them back to (C_out//n, C_in//n, ksize, ksize)
C_out, C_in, ksize, _ = J.shape
n = 2  # oscillator dimension

shape = (C_out // n, C_in // n, ksize, ksize)
c_R_reshaped = c_R.reshape(shape)
c_S_reshaped = c_S.reshape(shape)
alpha_reshaped = alpha.reshape(shape)
beta_reshaped = beta.reshape(shape)

In [ ]:
def subplot_spatial_slices(J, cmap='cividis', clim=None):
    """
    Visualize each (c_out, c_in) slice of J at every (h, w) position as a grid of subplots.

    Args:
        J: numpy array or torch tensor of shape (c_out, c_in, h, w)
    """
    if hasattr(J, "detach"):  # torch tensor
        J = J.detach().cpu().numpy()
    c_out, c_in, h, w = J.shape

    fig, axes = plt.subplots(h, w, figsize=(2*w, 2*h))
    if clim == None:
        vmin = J.min()
        vmax = J.max()
    else:
        vmin, vmax = clim[0], clim[1]

    for i in range(h):
        for j in range(w):
            ax = axes[i, j] if h > 1 and w > 1 else axes[max(i, j)]
            im = ax.imshow(J[:, :, i, j], cmap=cmap, vmin=vmin, vmax=vmax)
            ax.set_title(f'({i},{j})', fontsize=8)
            ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
subplot_spatial_slices(c_S_reshaped)

In [ ]:
subplot_spatial_slices(alpha_reshaped, clim=[-np.pi, np.pi], cmap='twilight')

In [ ]:
subplot_spatial_slices(beta_reshaped, clim=[-np.pi, np.pi], cmap='twilight')

In [ ]:
def subplot_kernel_slices(J, cmap='cividis', clim=None):
    """
    Visualize each (c_out, c_in) slice of J at every (h, w) position as a grid of subplots.

    Args:
        J: numpy array or torch tensor of shape (c_out, c_in, h, w)
    """
    if hasattr(J, "detach"):  # torch tensor
        J = J.detach().cpu().numpy()
    c_out, c_in, h, w = J.shape

    fig, axes = plt.subplots(h, w, figsize=(2*w, 2*h))
    if clim == None:
        vmin = J.min()
        vmax = J.max()
    else:
        vmin, vmax = clim[0], clim[1]

    for i in range(h):
        for j in range(w):
            ax = axes[i, j] if h > 1 and w > 1 else axes[max(i, j)]
            im = ax.imshow(J[:, :, i, j], cmap=cmap, vmin=vmin, vmax=vmax)
            ax.set_title(f'({i},{j})', fontsize=8)
            ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
def subplot_summary_statistics(J, stat_names=('mean', 'std', 'min', 'max'), cmap='viridis'):
    """
    Visualize summary statistics (mean, std, min, max) of the Frobenius norm of 2x2 blocks in J
    as 9x9 matrices (one for each kernel position).

    Args:
        J: numpy array or torch tensor of shape [C_out, C_in, H, W]
        stat_names: tuple of statistics to plot ('mean', 'std', 'min', 'max')
        cmap: colormap for imshow
    """
    if hasattr(J, "detach"):  # torch tensor
        J = J.detach().cpu().numpy()
    C_out, C_in, H, W = J.shape
    n = 2  # oscillator dimension

    num_out = C_out // n
    num_in = C_in // n

    # Compute Frobenius norm for each 2x2 block at each kernel position
    frob = np.zeros((num_out, num_in, H, W))
    for i in range(H):
        for j in range(W):
            for k in range(num_out):
                for l in range(num_in):
                    block = J[n*k:n*k+n, n*l:n*l+n, i, j]
                    frob[k, l, i, j] = np.linalg.norm(block, ord='fro')

    # Compute summary statistics for each (i, j) kernel position
    stats = {}
    stats['mean'] = frob.mean(axis=(0,1))
    stats['std']  = frob.std(axis=(0,1))
    stats['min']  = frob.min(axis=(0,1))
    stats['max']  = frob.max(axis=(0,1))

    # Plot each statistic as a 9x9 matrix
    n_stats = len(stat_names)
    fig, axes = plt.subplots(1, n_stats, figsize=(5*n_stats, 4))
    if n_stats == 1:
        axes = [axes]
    for idx, stat in enumerate(stat_names):
        im = axes[idx].imshow(stats[stat], cmap=cmap)
        axes[idx].set_title(f'{stat.capitalize()} Frobenius Norm\n(9x9 kernel grid)')
        axes[idx].set_xlabel('Kernel W')
        axes[idx].set_ylabel('Kernel H')
        plt.colorbar(im, ax=axes[idx], fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

In [ ]:
subplot_summary_statistics(c_R_reshaped)

In [ ]:
# # Visualize connectivity weight distributions
# fig, axes = plt.subplots(2, len(connectivity_weights), figsize=(5*len(connectivity_weights), 8))
# if len(connectivity_weights) == 1:
#     axes = axes.reshape(-1, 1)

# for i, conn in enumerate(connectivity_weights):
#     weight = conn['weight']
#     bias = conn['bias']
    
#     # Weight distribution
#     axes[0, i].hist(weight.flatten(), bins=50, alpha=0.7, density=True)
#     axes[0, i].set_title(f'Layer {i} Weight Distribution')
#     axes[0, i].set_xlabel('Weight Value')
#     axes[0, i].set_ylabel('Density')
#     axes[0, i].grid(True, alpha=0.3)
    
#     # Weight magnitude heatmap (first few filters)
#     # Show average over spatial dimensions for first 16 filters
#     if len(weight.shape) == 4:  # Conv weight [out_ch, in_ch, h, w]
#         weight_viz = np.mean(np.abs(weight[:16, :16]), axis=(2, 3))  # Average over spatial dims
#         im = axes[1, i].imshow(weight_viz, cmap='viridis', aspect='auto')
#         axes[1, i].set_title(f'Layer {i} Weight Magnitude (16x16 filters)')
#         axes[1, i].set_xlabel('Input Channel')
#         axes[1, i].set_ylabel('Output Channel')
#         axes[1, i].grid(False)
#         plt.colorbar(im, ax=axes[1, i])

# plt.tight_layout()
# plt.show()

# # Analyze kernel patterns
# print("\nAnalyzing learned kernel patterns:")
# for i, conn in enumerate(connectivity_weights):
#     weight = conn['weight']
#     if len(weight.shape) == 4:  # Conv kernels
#         kernel_size = weight.shape[2]
#         print(f"\nLayer {i} (kernel size {kernel_size}x{kernel_size}):")
        
#         # Compute average kernel
#         avg_kernel = np.mean(weight, axis=(0, 1))  # Average over input/output channels
#         print(f"  Average kernel center value: {avg_kernel[kernel_size//2, kernel_size//2]:.4f}")
#         print(f"  Average kernel edge/center ratio: {np.mean(avg_kernel[0, :]) / avg_kernel[kernel_size//2, kernel_size//2]:.4f}")

In [ ]:
convf = nn.Conv2d(128,64,9,1,4)

In [ ]:
tmp_out = convf(torch.randn(1,128,32,32))
tmp_out.shape

In [ ]:
convf.weight.shape

### Visualize Individual Kernels

In [ ]:
# def visualize_conv_kernels(connectivity_weights, layer_idx=0, num_kernels=16):
#     """Visualize individual convolutional kernels"""
#     if layer_idx >= len(connectivity_weights):
#         print(f"Layer {layer_idx} not found")
#         return
    
#     weight = connectivity_weights[layer_idx]['weight']  # [out_ch, in_ch, h, w]
#     out_ch, in_ch, h, w = weight.shape
    
#     # Select kernels to visualize
#     num_kernels = min(num_kernels, out_ch)
#     kernel_indices = np.linspace(0, out_ch-1, num_kernels, dtype=int)
    
#     fig, axes = plt.subplots(4, 4, figsize=(12, 12))
#     axes = axes.ravel()
    
#     for i, kernel_idx in enumerate(kernel_indices):
#         if i >= 16:
#             break
            
#         # Average over input channels for visualization
#         kernel = np.mean(weight[kernel_idx], axis=0)
        
#         im = axes[i].imshow(kernel, cmap='RdBu_r', vmin=-np.abs(kernel).max(), vmax=np.abs(kernel).max())
#         axes[i].set_title(f'Filter {kernel_idx}')
#         axes[i].axis('off')
#         plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)
    
#     # Hide unused subplots
#     for i in range(len(kernel_indices), 16):
#         axes[i].axis('off')
    
#     plt.suptitle(f'Layer {layer_idx} Convolutional Kernels ({h}x{w})', fontsize=16)
#     plt.tight_layout()
#     plt.show()

# # Visualize kernels for each layer
# for layer_idx in range(len(connectivity_weights)):
#     visualize_conv_kernels(connectivity_weights, layer_idx, num_kernels=16)